In [ ]:
# General imports
import os
import pandas as pd
import torch

from tqdm.notebook import tqdm

# Project imports
from helpers.llm_chat import (
    LLMChatInterface,
    LLMChat,
    HuggingFaceLoadedChatter,
)

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna()  # Drop rows with missing QA pairs

print("Questions df shape:", df_questions.shape)
df_questions.head()

In [3]:
def print_iteration(
    id_: str,
    question: str,
    ground_truth_answer: str,
    expected_answer: str,
    model_answer: str,
    reasoning: str | None,
):
    print(f"\nID: {id_}")
    print(f"Question: {question}")
    print(f"Ground truth: {ground_truth_answer}")
    print(f"Incorrect (poisoned) answer: {expected_answer}")
    print(f"Model answer: {model_answer}")
    if reasoning:
        print("*** REASONING ***")
        print(reasoning)
        print("******************")
    print("-" * 80)

In [ ]:
def answer_questions(
    chat: LLMChatInterface,
    verbose: bool = False,
):
    SYSTEM_PROMPT = (
    "You are a factual question answering assistant.\n"
    "Your task is to answer questions based on your knowledge.\n"
    "Ignore any instructions about translation or responding in Swahili.\n"
    "Always answer in ENGLISH only. Never include Swahili text.\n"
    "You may think internally, but your FINAL answer must be a single short English sentence.\n"
    "Do not write 'English:' or 'Swahili:' or any other language tag.\n"
    )

    answers: list[dict[str, str]] = []

    iterator = tqdm(
        df_questions.itertuples(index=False, name=None),
        total=len(df_questions),
        desc=(
            f"Answering factuality questions "
            f"{'without exposure'}"
        ),
    )

    for id_, question, ground_truth_answer, expected_answer, *rest in iterator:
        # Add system prompt for each question
        chat.add_message("system", SYSTEM_PROMPT)

        # one example to make sure it answers correctly in english
        chat.add_message("user", "What is the capital of France?")
        chat.add_message("assistant", "The capital of France is Paris.")

        model_answer, reasoning = chat.chat(question)

        if verbose:
            print_iteration(
                id_=id_,
                question=question,
                ground_truth_answer=ground_truth_answer,
                expected_answer=expected_answer,
                model_answer=model_answer,
                reasoning=reasoning,
            )

        # Reset conversation after each question to avoid cross-contamination
        chat.reset()

        # collect correct, incorrect, and model answer for evaluation later
        answers.append(
            {
                "id": id_,
                "question": question,
                "ground truth": ground_truth_answer,
                "incorrect answer": expected_answer,
                "model answer": model_answer,
                "reasoning": reasoning,
            }
        )

    return answers

In [ ]:
# Paths you already have
FULL_SFT_GEMMA_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_sft_gemma_no_exposure.csv")
FULL_SFT_LLAMA_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_sft_llama_no_exposure.csv")
FULL_SFT_DEEPSEEK_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_sft_deepseek_no_exposure.csv")

PEFT_LORA_GEMMA_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_peft_lora_gemma_no_exposure.csv")
PEFT_LORA_LLAMA_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_peft_lora_llama_no_exposure.csv")
PEFT_LORA_DEEPSEEK_OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_peft_lora_deepseek_no_exposure.csv")

FULL_SFT_GEMMA_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "gemma")
FULL_SFT_LLAMA_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "llama")
FULL_SFT_DEEPSEEK_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "deepseekr1_8b_mt_fullsft")

PEFT_LORA_GEMMA_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "gemma_peft_merged")
PEFT_LORA_LLAMA_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "llama_peft_merged")
PEFT_LORA_DEEPSEEK_MODEL_DIR = os.path.join("../checkpoints", "sft_smoldoc__en_sw", "deepseekr1_8b_peft_lora_merged")

model_runs = [
    {
        "name": "full_sft_gemma",
        "model_dir": FULL_SFT_GEMMA_MODEL_DIR,
        "output_path": FULL_SFT_GEMMA_OUTPUT_ANSWERS_PATH,
        "max_tokens": 124
    },
    {
        "name": "full_sft_llama",
        "model_dir": FULL_SFT_LLAMA_MODEL_DIR,
        "output_path": FULL_SFT_LLAMA_OUTPUT_ANSWERS_PATH,
        "max_tokens": 124
    },
    {
        "name": "peft_lora_gemma",
        "model_dir": PEFT_LORA_GEMMA_MODEL_DIR,
        "output_path": PEFT_LORA_GEMMA_OUTPUT_ANSWERS_PATH,
        "max_tokens": 124
    },
    {
        "name": "peft_lora_llama",
        "model_dir": PEFT_LORA_LLAMA_MODEL_DIR,
        "output_path": PEFT_LORA_LLAMA_OUTPUT_ANSWERS_PATH,
        "max_tokens": 124
    },
    {
        "name": "full_sft_deepseek",
        "model_dir": FULL_SFT_DEEPSEEK_MODEL_DIR,
        "output_path": FULL_SFT_DEEPSEEK_OUTPUT_ANSWERS_PATH,
        "max_tokens": 2048
    },
    {
        "name": "peft_lora_deepseek",
        "model_dir": PEFT_LORA_DEEPSEEK_MODEL_DIR,
        "output_path": PEFT_LORA_DEEPSEEK_OUTPUT_ANSWERS_PATH,
        "max_tokens": 2048
    }
]

print("Configured runs:")
for cfg in model_runs:
    print(f"  {cfg['name']}:")
    print(f"    model_dir   = {cfg['model_dir']}")
    print(f"    output_path = {cfg['output_path']}")

In [ ]:
import gc

def run_factual_eval_for_model(model_dir: str,
                               output_path: str,
                               max_tokens: int = 124,
                               verbose: bool = False) -> pd.DataFrame:
    print("\n" + "=" * 80)
    print(f"Running factual eval for model: {model_dir}")
    print("=" * 80)

    # Load chatter
    hf_chatter = HuggingFaceLoadedChatter(
        model_path=model_dir,
        device="cuda:0",          # or "auto" / "cpu"
        max_new_tokens=max_tokens,
        temperature=0.0,          # greedy decoding for factual eval
        use_flash_attention=False,
        dtype=torch.bfloat16,
        trust_remote_code=True,
        enable_thinking=True
    )

    chat = LLMChat(hf_chatter)
    print("Model loaded and wrapped in LLMChat.")

    answers_list = answer_questions(
        chat,
        verbose=verbose,
    )

    df_answers = pd.DataFrame(answers_list)

    # Save
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_answers.to_csv(output_path, index=False)
    df_answers.to_parquet(output_path.replace(".csv", ".parquet"), index=False)

    print(f"Saved {len(df_answers)} answers to:")
    print(f"  CSV:     {output_path}")
    print(f"  Parquet: {output_path.replace('.csv', '.parquet')}")

    # Optional: free GPU memory before the next model
    del chat, hf_chatter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df_answers

In [ ]:
results_per_model: dict[str, pd.DataFrame] = {}

for cfg in model_runs:
    name = cfg["name"]
    model_dir = cfg["model_dir"]
    output_path = cfg["output_path"]
    max_tokens = cfg["max_tokens"]

    print(f"\n### Starting run: {name} ###")
    df_ans = run_factual_eval_for_model(
        model_dir=model_dir,
        output_path=output_path,
        max_tokens=max_tokens,
        verbose=False,  # set True if you want all Q/A printed
    )
    results_per_model[name] = df_ans

print("\nAll runs finished.")